# spatioloji_s — Basic Workflow Tutorial

This notebook covers the complete single-cell processing pipeline for image-based spatial transcriptomics data (CosMx, MERFISH, Xenium) using **spatioloji_s**.

## Pipeline overview

```
Load data
  └─ Quick summary
  └─ Quality control (cells + genes + FOVs)
     └─ Normalization (library size → log1p → scale)
        └─ Highly variable gene selection
           └─ Dimensionality reduction (PCA → UMAP)
              └─ Clustering (Leiden)
                 └─ Visualization & marker analysis
                    └─ Save
```

**New in this tutorial** (compared to the basic processing notebook):
- `compare_hvg_methods` — Jaccard overlap and consensus across 4 HVG methods
- `find_optimal_clusters(method='all')` — consensus k from 3 geometric metrics
- `leiden_resolution_sweep` — stability-based Leiden resolution selection (ARI)
- `assess_clustering_quality` — post-hoc silhouette / DB / CH evaluation

## 0. Installation

```bash
pip install spatioloji-s
pip install "spatioloji-s[clustering]"   # Leiden (leidenalg + igraph)
pip install "spatioloji-s[reduction]"    # UMAP
```

In [ ]:
import os
import matplotlib.pyplot as plt
import spatioloji_s as sj

print(f"spatioloji_s version: {sj.__version__}")

---
## 1. Load Data

### Option A — Load from raw files

```python
sp = sj.spatioloji.from_files(
    polygons_path      = "path/to/polygons.csv",
    cell_meta_path     = "path/to/cell_metadata.csv",
    expression_path    = "path/to/expression.npz",
    fov_positions_path = "path/to/fov_positions.csv",
    images_folder      = "path/to/images/",
)
```

### Option B — Load a saved object (recommended after QC)

### File format reference

The five input files and one folder expected by `sj.spatioloji.from_files()` are described below.
All CSV files must be UTF-8 encoded with a header row.

---

#### 1. `cell_metadata.csv` — per-cell metadata

Each row is one cell.

| Column | Type | Required | Default name | Description |
|---|---|---|---|---|
| `cell` | str / int | **Yes** | `cell_id_col` | Unique cell identifier (master index) |
| `fov` | int | **Yes** | `fov_id_col` | Field-of-view ID the cell belongs to |
| `x_local_px` | float | **Yes** | `x_local_col` | X centroid in local FOV pixel space |
| `y_local_px` | float | **Yes** | `y_local_col` | Y centroid in local FOV pixel space |
| `x_global_px` | float | **Yes** | `x_global_col` | X centroid in global (stitched) pixel space |
| `y_global_px` | float | **Yes** | `y_global_col` | Y centroid in global (stitched) pixel space |
| `Area` | float | No | — | Cell area in pixels (used by QC) |
| *any others* | any | No | — | Cell type annotations, batch labels, etc. |

> **CosMx native export** provides these as `cell_ID`, `fov`, `CenterX_local_px`,
> `CenterY_local_px`, `CenterX_global_px`, `CenterY_global_px` — rename or use
> `SpatiolojiConfig` to map non-default column names (see below).

Minimal example:

```
cell,fov,x_local_px,y_local_px,x_global_px,y_global_px
cell_001,1,123.4,456.7,10123.4,20456.7
cell_002,1,200.1,500.3,10200.1,20500.3
cell_003,2,88.5,312.9,21088.5,15312.9
```

---

#### 2. `expression.npz` — gene expression matrix

A **NumPy compressed archive** (`.npz`) containing three arrays:

| Key | Shape | dtype | Description |
|---|---|---|---|
| `data` | `(n_cells, n_genes)` | float32 | Raw transcript counts (sparse or dense) |
| `cell_ids` | `(n_cells,)` | str / int | Cell IDs — must match `cell` column in cell_meta |
| `gene_names` | `(n_genes,)` | str | Gene names |

Create from a dense count matrix:

```python
import numpy as np

np.savez_compressed(
    "expression.npz",
    data       = counts_matrix.astype(np.float32),   # shape (n_cells, n_genes)
    cell_ids   = cell_id_array,
    gene_names = gene_name_array,
)
```

A plain **CSV** (`expression.csv`, rows = cells, columns = genes, index = cell IDs)
is also accepted but not recommended for large panels (>500 genes / >50k cells).

---

#### 3. `polygons.csv` — cell boundary vertices

Each row is **one vertex** of one cell's polygon.

> **Both local and global coordinates are required** in this file.
> `to_geopandas(coord_type='global')` (default for all polygon spatial analysis)
> reads `x_global_px` / `y_global_px` directly from this DataFrame.
> Local coordinates are used for single-FOV visualisation and FOV-level analysis.

| Column | Type | Required | Description |
|---|---|---|---|
| `cell` | str / int | **Yes** | Cell identifier — must match cell_meta |
| `fov` | int | **Yes** | FOV ID |
| `x_local_px` | float | **Yes** | Vertex X in local FOV pixel space |
| `y_local_px` | float | **Yes** | Vertex Y in local FOV pixel space |
| `x_global_px` | float | **Yes** | Vertex X in global (stitched) pixel space |
| `y_global_px` | float | **Yes** | Vertex Y in global (stitched) pixel space |

The loader groups rows by `cell` and constructs Shapely `Polygon` objects.
Cells with fewer than 3 vertices are dropped with a warning.

If your source only has local coordinates, derive global ones by adding the FOV offset:

```python
import pandas as pd

fov_pos = pd.read_csv("fov_positions.csv").set_index("fov")
poly = pd.read_csv("polygons_local_only.csv")

# Vectorised: map each vertex's FOV offset
poly["x_global_px"] = poly["x_local_px"] + poly["fov"].map(fov_pos["x_global_px"])
poly["y_global_px"] = poly["y_local_px"] + poly["fov"].map(fov_pos["y_global_px"])

poly.to_csv("polygons.csv", index=False)
```

Minimal example with both coordinate sets:

```
cell,fov,x_local_px,y_local_px,x_global_px,y_global_px
cell_001,1,100.0,200.0,10100.0,20200.0
cell_001,1,150.0,250.0,10150.0,20250.0
cell_001,1,100.0,300.0,10100.0,20300.0
cell_002,1,400.0,500.0,10400.0,20500.0
cell_002,1,420.0,530.0,10420.0,20530.0
cell_002,1,390.0,540.0,10390.0,20540.0
```

> **CosMx native format** ships polygons in long-form vertex layout but with local
> coordinates only. Use the snippet above to add global coords.
> **MERFISH** provides per-cell WKT strings — explode to vertex rows, then add global coords.

---

#### 4. `fov_positions.csv` — global FOV offsets

Used to stitch local pixel coordinates into global space and to locate images on disk.

| Column | Type | Required | Description |
|---|---|---|---|
| `fov` | int | **Yes** | FOV ID (must match `fov` in cell_meta) |
| `x_global_px` | float | **Yes** | X offset of FOV origin in global pixel space |
| `y_global_px` | float | **Yes** | Y offset of FOV origin in global pixel space |

Minimal example:

```
fov,x_global_px,y_global_px
1,0.0,0.0
2,10000.0,0.0
3,0.0,10000.0
```

> **CosMx** exports this as `fov_positions_file.csv` with columns
> `FOV`, `X_mm`, `Y_mm`. Convert mm to pixels by dividing by the pixel size
> (typically 0.18 um/px: multiply mm by 1000/0.18).

---

#### 5. `images/` folder — FOV microscopy images

The loader scans the folder and matches files using the pattern:

```
CellComposite_F{fov_id}.{ext}
```

where `{fov_id}` is the zero-padded or plain integer FOV ID and `{ext}` is one of:
`jpg`, `jpeg`, `png`, `tif`, `tiff`.

Example folder layout:

```
images/
  CellComposite_F001.jpg
  CellComposite_F002.jpg
  CellComposite_F003.jpg
```

Images are **lazy-loaded** on first access per FOV and cached with an LRU cache
(default size 8 FOVs). Pass `images_folder=None` if you have no images.

---

#### 6. `SpatiolojiConfig` — column name customisation

If your files use non-default column names, pass a `SpatiolojiConfig` object.
The same config applies to **both** cell_meta and polygons columns.

```python
from spatioloji_s.data.config import SpatiolojiConfig

cfg = SpatiolojiConfig(
    cell_id_col  = "cell_ID",           # default: "cell"
    fov_id_col   = "fov",               # default: "fov"
    x_local_col  = "CenterX_local_px",  # default: "x_local_px"  (used in both cell_meta + polygons)
    y_local_col  = "CenterY_local_px",  # default: "y_local_px"
    x_global_col = "CenterX_global_px", # default: "x_global_px"
    y_global_col = "CenterY_global_px", # default: "y_global_px"
)

sp = sj.spatioloji.from_files(
    polygons_path      = "polygons.csv",
    cell_meta_path     = "cell_metadata.csv",
    expression_path    = "expression.npz",
    fov_positions_path = "fov_positions.csv",
    images_folder      = "images/",
    config             = cfg,
)
```

---

#### Platform-specific notes

| Platform | cell_meta | polygons | expression | fov_positions |
|---|---|---|---|---|
| **CosMx** | `*_metadata_file.csv` (rename cols or use config) | `*_polygons.csv` (local coords only — add global) | `*_exprMat_file.csv` (convert to .npz) | `*fov_positions_file.csv` (convert mm to px) |
| **MERFISH** | `cell_by_gene.csv` metadata columns | WKT strings (explode to vertex rows, add global) | `cell_by_gene.csv` count block | Reconstruct from global centroid coords |
| **Xenium** | `cells.csv` | `cell_boundaries.csv` (local coords only — add global) | `cell_feature_matrix/` (10x sparse, use `scipy.io.mmread`) | `experiment.xenium` JSON (extract FOV offsets) |

### Helper: convert CosMx native export

If you are working with CosMx data, use the snippet below to prepare the files
before calling `from_files()`. Adjust paths and the pixel-size constant for your
instrument (typical CosMx: 0.18 um/pixel).

```python
import numpy as np
import pandas as pd

# ---------- paths (adjust to your CosMx export folder) ----------
cosmx_dir      = "path/to/CosMx_export/"
cell_meta_raw  = cosmx_dir + "RunSummary/expt_metadata_file.csv"
expression_raw = cosmx_dir + "RunSummary/expt_exprMat_file.csv"
polygons_raw   = cosmx_dir + "RunSummary/expt_polygons.csv"
fov_pos_raw    = cosmx_dir + "RunSummary/expt_fov_positions_file.csv"

UM_PER_PX = 0.18   # instrument-specific; adjust as needed

# ---------- cell metadata ----------
meta = pd.read_csv(cell_meta_raw)
meta = meta.rename(columns={
    "cell_ID"            : "cell",
    "CenterX_local_px"   : "x_local_px",
    "CenterY_local_px"   : "y_local_px",
    "CenterX_global_px"  : "x_global_px",
    "CenterY_global_px"  : "y_global_px",
})
meta.to_csv("cell_metadata.csv", index=False)

# ---------- expression matrix (.npz) ----------
expr_df   = pd.read_csv(expression_raw, index_col=0)   # rows = cells, cols = genes
np.savez_compressed(
    "expression.npz",
    data       = expr_df.values.astype(np.float32),
    cell_ids   = np.array(expr_df.index.astype(str)),
    gene_names = np.array(expr_df.columns.astype(str)),
)

# ---------- FOV positions (mm -> pixels) ----------
fov_pos = pd.read_csv(fov_pos_raw)
fov_pos = fov_pos.rename(columns={"FOV": "fov"})
fov_pos["x_global_px"] = fov_pos["X_mm"] * 1000.0 / UM_PER_PX
fov_pos["y_global_px"] = fov_pos["Y_mm"] * 1000.0 / UM_PER_PX
fov_pos = fov_pos[["fov", "x_global_px", "y_global_px"]]
fov_pos.to_csv("fov_positions.csv", index=False)

# ---------- polygons (local coords only in CosMx -> add global) ----------
# CosMx polygons are in long-form vertex layout but only have local coordinates.
# Global coords = local coords + FOV origin offset.
poly = pd.read_csv(polygons_raw)
poly = poly.rename(columns={
    "cell_ID"    : "cell",
    "x_local_px" : "x_local_px",   # already correct name in most exports
    "y_local_px" : "y_local_px",
})
fov_index = fov_pos.set_index("fov")
poly["x_global_px"] = poly["x_local_px"] + poly["fov"].map(fov_index["x_global_px"])
poly["y_global_px"] = poly["y_local_px"] + poly["fov"].map(fov_index["y_global_px"])
poly.to_csv("polygons.csv", index=False)

print("Files ready:")
print("  cell_metadata.csv, expression.npz, fov_positions.csv, polygons.csv")
print("  Images should be in a folder as CellComposite_F{fov}.jpg")
```

In [ ]:
# Load a previously saved spatioloji object
sp = sj.spatioloji.from_pickle("my_data/raw_spatioloji.pkl")

# For this tutorial we assume CosMx data with columns:
#   cell_meta: fov, cell_ID, Area, CenterX_local_px, CenterY_local_px,
#              CenterX_global_px, CenterY_global_px, ...
print(sp)

---
## 2. Quick Summary

In [ ]:
# High-level overview: cells, genes, FOVs, available layers
sj.data.utils.quick_summary(sp)

In [ ]:
# Inspect metadata columns
print("Cell meta columns:", sp.cell_meta.columns.tolist())
print("Gene meta columns:", sp.gene_meta.columns.tolist())
print(f"\nCells : {sp.n_cells:,}")
print(f"Genes : {sp.n_genes:,}")
print(f"FOVs  : {sp.n_fovs}")

---
## 3. Quality Control

The `spatioloji_qc` class provides a modular QC pipeline:

| Step | What it checks |
|---|---|
| `qc_negative_probes` | Outlier FOVs with excess negative-probe signal (Grubbs test) |
| `qc_cell_area` | Outlier cell sizes |
| `qc_cell_metrics` | Total counts, n_genes, % mt, % NegProbe per cell |
| `qc_fov_metrics` | Per-FOV cell counts and transcript density |
| `filter_cells` | Apply threshold-based cell filtering |
| `filter_genes` | Filter genes by expression vs. NegProbe baseline |

In [ ]:
# Configure QC thresholds
qc_config = sj.QCConfig(
    # Cell filters
    total_counts_min        = 20,      # min transcripts per cell
    ratio_counts_genes_min  = 1.0,     # min counts/gene ratio
    pct_counts_mt_max       = 0.25,    # max 25% mitochondrial
    pct_counts_neg_max      = 0.1,     # max 10% NegProbe
    # Gene filters
    gene_filter_method      = "percentile",
    gene_percentile_threshold = 50,    # keep genes above 50th pct of NegProbe
    output_dir              = "./qc_output/",
    save_plots              = True,
)

In [ ]:
# Initialise QC (automatically computes per-cell metrics)
qc = sj.spatioloji_qc(sp, config=qc_config)

In [ ]:
# Run individual QC steps with visualisation
qc.qc_negative_probes(plot=True)
qc.qc_cell_area(area_column="Area", plot=True)
qc.qc_cell_metrics(plot=True)
qc.qc_fov_metrics(plot=True)

In [ ]:
# Filter cells
cell_mask = qc.filter_cells()   # adds QC_pass column to sp.cell_meta

# Filter genes (NegProbe-aware)
gene_mask = qc.filter_genes(plot=True)

In [ ]:
# Apply filters → returns a new, filtered spatioloji object
sp_filtered = qc.apply_filters()

print(f"Before QC: {sp.n_cells:,} cells, {sp.n_genes:,} genes")
print(f"After QC : {sp_filtered.n_cells:,} cells, {sp_filtered.n_genes:,} genes")

In [ ]:
# Save filtered object so QC does not need to be re-run
os.makedirs("my_data", exist_ok=True)
sp_filtered.to_pickle("my_data/filtered_spatioloji.pkl")

# Work with the filtered object from here on
sp = sp_filtered

---
## 4. Normalization

Standard workflow for image-based ST:

```
raw counts
  → normalize_total  (library-size normalization, target = 10,000)
  → log_transform    (log1p, stabilises variance)
  → scale            (z-score per gene, for PCA input only)
```

Each step is stored as a named **layer** so no data is lost.

In [ ]:
# Step 1 — library-size normalization
sj.processing.normalize_total(sp, target_sum=1e4, inplace=True)
# Adds layer: 'normalized_counts'

In [ ]:
# Step 2 — log1p transform
sj.processing.log_transform(sp, layer="normalized_counts", inplace=True)
# Adds layer: 'log_normalized'

In [ ]:
# Step 3 — z-score scaling (used only as PCA input; do NOT use for DE or violin plots)
sj.processing.scale(sp, layer="log_normalized", max_value=10.0, inplace=True)
# Adds layer: 'scaled'

In [ ]:
# Confirm all layers are present
print("Available layers:", sp.list_layers())

---
## 5. Highly Variable Gene (HVG) Selection

HVG selection reduces noise before PCA by focusing on genes that carry biological signal.

### 5a. Single-method selection

In [ ]:
# Recommended default: Seurat v3 VST (polynomial log-log regression)
# Run on normalized_counts (BEFORE log, as count data is expected)
sj.processing.highly_variable_genes(
    sp,
    layer       = "normalized_counts",
    method      = "seurat_v3",
    n_top_genes = 2000,
    inplace     = True,
)

n_hvg = sp.gene_meta["highly_variable"].sum()
print(f"\nHVGs selected: {n_hvg}")

### 5b. Compare HVG methods

`compare_hvg_methods` runs multiple methods and reports pairwise Jaccard similarity,
helping you understand how consistent the gene lists are across methods and identify
a **consensus** set selected by the majority of methods.

In [ ]:
hvg_comparison = sj.processing.compare_hvg_methods(
    sp,
    methods     = ["seurat", "seurat_v3", "pearson_residuals", "deviance"],
    n_top_genes = 2000,
    layer       = "normalized_counts",
)

In [ ]:
# Genes selected by all 4 methods (most robust HVG set)
consensus_genes = hvg_comparison.index[hvg_comparison["consensus"]].tolist()
print(f"Consensus HVGs (majority vote): {len(consensus_genes)}")

# Genes selected by every single method
universal_genes = hvg_comparison.index[
    hvg_comparison["n_methods_selected"] == 4
].tolist()
print(f"Universal HVGs (all 4 methods): {len(universal_genes)}")

In [ ]:
# View the comparison table (True = selected by that method)
hvg_comparison.sort_values("n_methods_selected", ascending=False).head(20)

In [ ]:
# Tip: if you want to use the consensus set for downstream analysis,
# write it back to gene_meta and re-run scale + PCA on it:
sp.gene_meta["highly_variable"] = hvg_comparison["consensus"].reindex(
    sp.gene_meta.index, fill_value=False
).values
print(f"Updated HVG count: {sp.gene_meta['highly_variable'].sum()}")

---
## 6. Dimensionality Reduction

### 6a. PCA

In [ ]:
# PCA on scaled expression of HVGs
sj.processing.pca(
    sp,
    layer                = "scaled",
    use_highly_variable  = True,   # uses sp.gene_meta["highly_variable"]
    n_comps              = 50,
    random_state         = 42,
    inplace              = True,
)
# Stores X_pca in sp._embeddings and PC1..PC50 in sp.cell_meta

In [ ]:
# Elbow plot — variance explained by each PC
sj.processing.plot_pca_variance(sp, n_pcs=30)

### 6b. UMAP

In [ ]:
# UMAP on top PCs (requires: pip install "spatioloji-s[reduction]")
sj.processing.umap(
    sp,
    use_pca      = True,
    n_pcs        = 20,     # number of PCs to use as input
    n_neighbors  = 30,
    min_dist     = 0.3,
    random_state = 42,
    inplace      = True,
)
# Stores UMAP1 / UMAP2 in sp.cell_meta and X_umap in sp._embeddings

---
## 7. Clustering

### 7a. Find optimal K (for K-Means)

Use `method='all'` to run silhouette, Davies-Bouldin, and Calinski-Harabasz
simultaneously and get a consensus recommendation.

In [ ]:
k_result = sj.processing.find_optimal_clusters(
    sp,
    layer        = "scaled",
    method       = "all",       # consensus of sil + DB + CH
    k_range      = (2, 15),
    n_pcs        = 20,
    sample_size  = 5000,        # subsample for silhouette speed
    random_state = 42,
)

print(f"\nConsensus optimal K : {k_result['optimal_k']}")
print(f"Best K by silhouette : {k_result['optimal_k_silhouette']}")
print(f"Best K by Davies-Bouldin : {k_result['optimal_k_davies_bouldin']}")
print(f"Best K by Calinski-Harabasz : {k_result['optimal_k_calinski_harabasz']}")

In [ ]:
# Plot all metric curves for visual inspection
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
ks = k_result["k_values"]

axes[0].plot(ks, k_result["silhouette_scores"], "o-", color="steelblue")
axes[0].set(title="Silhouette Score", xlabel="K", ylabel="Score (higher=better)")
axes[0].axvline(k_result["optimal_k_silhouette"], color="red", linestyle="--", label=f"K={k_result['optimal_k_silhouette']}")
axes[0].legend()

axes[1].plot(ks, k_result["davies_bouldin_scores"], "o-", color="coral")
axes[1].set(title="Davies-Bouldin Index", xlabel="K", ylabel="Index (lower=better)")
axes[1].axvline(k_result["optimal_k_davies_bouldin"], color="red", linestyle="--", label=f"K={k_result['optimal_k_davies_bouldin']}")
axes[1].legend()

axes[2].plot(ks, k_result["calinski_harabasz_scores"], "o-", color="seagreen")
axes[2].set(title="Calinski-Harabasz Index", xlabel="K", ylabel="Index (higher=better)")
axes[2].axvline(k_result["optimal_k_calinski_harabasz"], color="red", linestyle="--", label=f"K={k_result['optimal_k_calinski_harabasz']}")
axes[2].legend()

plt.suptitle("Optimal K Assessment", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("my_data/optimal_k_metrics.pdf", bbox_inches="tight")
plt.show()

### 7b. Leiden resolution sweep (stability-based)

`leiden_resolution_sweep` runs Leiden `n_runs` times at each resolution and reports
the mean pairwise **ARI** (Adjusted Rand Index) across runs as a stability measure.
Pick a resolution inside a **stable plateau** (high ARI) at your desired granularity.

In [ ]:
import numpy as np

sweep_df = sj.processing.leiden_resolution_sweep(
    sp,
    resolutions = list(np.arange(0.2, 1.6, 0.2)),  # [0.2, 0.4, 0.6, ..., 1.4]
    n_runs      = 5,     # runs per resolution (more = more stable estimate)
    n_pcs       = 20,
    n_neighbors = 15,
    random_state = 42,
)

sweep_df

In [ ]:
# Visualise stability and cluster count together
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.plot(sweep_df["resolution"], sweep_df["mean_ari"], "o-", color="steelblue", label="Mean ARI (stability)")
ax1.fill_between(
    sweep_df["resolution"],
    sweep_df["mean_ari"] - sweep_df["std_ari"],
    sweep_df["mean_ari"] + sweep_df["std_ari"],
    alpha=0.2, color="steelblue"
)
ax2.plot(sweep_df["resolution"], sweep_df["n_clusters_mean"], "s--", color="coral", label="N clusters")

ax1.set_xlabel("Resolution")
ax1.set_ylabel("Mean ARI (stability)", color="steelblue")
ax2.set_ylabel("N clusters", color="coral")
ax1.set_title("Leiden Resolution Sweep")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")
plt.tight_layout()
plt.savefig("my_data/leiden_resolution_sweep.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Pick the most stable resolution (or override based on biology)
optimal_res = float(sweep_df.loc[sweep_df["mean_ari"].idxmax(), "resolution"])
print(f"Most stable resolution: {optimal_res}")

# Override if you prefer a specific number of clusters
# optimal_res = 0.5

### 7c. Run Leiden clustering

In [ ]:
sj.processing.leiden_clustering(
    sp,
    layer               = "scaled",
    use_highly_variable = True,
    resolution          = optimal_res,
    n_neighbors         = 15,
    n_pcs               = 50,
    use_pca             = True,
    random_state        = 42,
    output_column       = "leiden",
    inplace             = True,
)

n_clusters = sp.cell_meta["leiden"].nunique()
print(f"\nLeiden clusters: {n_clusters}")
print(sp.cell_meta["leiden"].value_counts().sort_index())

### 7d. Assess clustering quality

In [ ]:
metrics = sj.processing.assess_clustering_quality(
    sp,
    cluster_col = "leiden",
    layer       = "scaled",
    n_pcs       = 20,
    sample_size = 5000,
)

print("\nQuality metrics:")
for k, v in metrics.items():
    if k != "cluster_sizes":
        print(f"  {k}: {v}")

In [ ]:
# Optional: try multiple resolutions and compare quality
import pandas as pd

quality_rows = []
for res in [0.3, 0.5, 0.7, 1.0]:
    sj.processing.leiden_clustering(
        sp, resolution=res, output_column=f"leiden_{res}", inplace=True
    )
    m = sj.processing.assess_clustering_quality(
        sp, cluster_col=f"leiden_{res}", layer="scaled", n_pcs=20, sample_size=3000
    )
    quality_rows.append({"resolution": res, **{k: v for k, v in m.items() if k != "cluster_sizes"}})

quality_df = pd.DataFrame(quality_rows)
quality_df

---
## 8. Visualization

### 8a. UMAP coloured by cluster

In [ ]:
import os
os.makedirs("my_plots", exist_ok=True)

# Categorical: colour by Leiden cluster
sj.visualization.plot_umap(
    sp,
    color_by  = "leiden",
    palette   = "tab20",
    point_size = 4.0,
    title     = "UMAP — Leiden clusters",
    show      = True,
    save_path = "my_plots/UMAP_leiden.pdf",
)

In [ ]:
# Continuous: colour by a QC metric
sj.visualization.plot_umap(
    sp,
    color_by  = "total_counts",
    color_map = "viridis",
    title     = "UMAP — Total counts",
    show      = True,
    save_path = "my_plots/UMAP_total_counts.pdf",
)

In [ ]:
# Gene expression overlay (uses log_normalized layer by default)
sj.visualization.plot_umap(
    sp,
    gene      = "EPCAM",      # replace with a gene in your panel
    layer     = "log_normalized",
    color_map = "Reds",
    title     = "UMAP — EPCAM expression",
    show      = True,
    save_path = "my_plots/UMAP_EPCAM.pdf",
)

### 8b. Multi-panel UMAP grid

In [ ]:
# Plot multiple genes and metadata columns in one figure
# Replace gene names with genes present in your panel
sj.visualization.plot_umap_grid(
    sp,
    genes    = ["EPCAM", "CD3D", "CD14", "MS4A1"],  # adjust to your panel
    features = ["total_counts", "n_genes_by_counts"],
    ncols    = 3,
    layer    = "log_normalized",
    show     = True,
    save_path = "my_plots/UMAP_grid.pdf",
)

### 8c. Violin plots — marker genes per cluster

In [ ]:
# Single gene
sj.visualization.plot_violin(
    sp,
    genes    = "EPCAM",
    group_by = "leiden",
    layer    = "log_normalized",
    show     = True,
    save_path = "my_plots/violin_EPCAM.pdf",
)

In [ ]:
# Multiple genes as a grid of violins
marker_genes = ["EPCAM", "CD3D", "CD14", "MS4A1", "CD68", "DCN"]

sj.visualization.plot_violin(
    sp,
    genes    = marker_genes,
    group_by = "leiden",
    layer    = "log_normalized",
    show     = True,
    save_path = "my_plots/violin_markers.pdf",
)

### 8d. Heatmap — mean expression per cluster

In [ ]:
# Arrange genes by cell type for interpretable heatmap
gene_order = [
    # Epithelial
    "EPCAM", "KRT18", "KRT19",
    # T cells
    "CD3D", "CD3E", "CD8A", "CD4",
    # B cells
    "MS4A1", "CD79A",
    # Myeloid
    "CD14", "CD68", "LYZ",
    # Fibroblast
    "DCN", "COL1A1",
]
# Filter to genes actually in the panel
gene_order = [g for g in gene_order if g in sp.gene_index.tolist()]

cluster_order = [str(i) for i in sorted(sp.cell_meta["leiden"].unique())]

sj.visualization.plot_heatmap(
    sp,
    genes       = gene_order,
    group_by    = "leiden",
    gene_order  = gene_order,
    group_order = cluster_order,
    scale       = "row",          # z-score per gene across clusters
    layer       = "log_normalized",
    show        = True,
    save_path   = "my_plots/heatmap_markers.pdf",
)

### 8e. Dot plot — expression fraction + intensity

In [ ]:
sj.visualization.plot_dotplot(
    sp,
    genes      = gene_order,
    gene_order = gene_order,
    group_by   = "leiden",
    layer      = "log_normalized",
    show       = True,
    save_path  = "my_plots/dotplot_markers.pdf",
)

---
## 9. Save the Final Object

In [ ]:
sp.to_pickle("my_data/processed_spatioloji.pkl")
print("Saved.")

In [ ]:
# Final summary
print("=" * 50)
print("Processed object summary")
print("=" * 50)
print(f"  Cells   : {sp.n_cells:,}")
print(f"  Genes   : {sp.n_genes:,}")
print(f"  FOVs    : {sp.n_fovs}")
print(f"  Layers  : {sp.list_layers()}")
print(f"  Leiden clusters : {sp.cell_meta['leiden'].nunique()}")
print("\nNext steps:")
print("  → spatial_analysis.ipynb  — neighbourhood enrichment, Moran's I, Ripley K")
print("  → ccc.ipynb               — cell-cell communication (3-layer polygon framework)")

---
## Appendix: Processing Parameter Reference

### Normalization layers

| Layer name | Created by | Use for |
|---|---|---|
| *(main matrix)* | raw input | HVG deviance / Pearson residuals |
| `normalized_counts` | `normalize_total` | HVG seurat / seurat_v3 / pearson_residuals |
| `log_normalized` | `log_transform` | Visualization, violin, heatmap, dotplot |
| `scaled` | `scale` | PCA input **only** |

### HVG method guide

| Method | Input layer | Best for |
|---|---|---|
| `seurat` | normalized_counts | Legacy; wide compatibility |
| `seurat_v3` | normalized_counts | Standard; best overall for ST |
| `pearson_residuals` | raw counts | Count data; no pre-normalization |
| `deviance` | raw counts | Sparse panels (CosMx/Xenium <1000 genes) |
| `spatial_moran` | log_normalized | Spatially variable genes |

### Clustering quick reference

| Function | Purpose |
|---|---|
| `find_optimal_clusters(method='all')` | Consensus K for K-Means |
| `leiden_resolution_sweep` | Stability sweep for Leiden |
| `leiden_clustering` | Main graph-based clustering |
| `kmeans_clustering` | K-Means (good for known K) |
| `hierarchical_clustering` | Dendrogram-based |
| `assess_clustering_quality` | Post-hoc silhouette / DB / CH |
| `spatially_constrained_clustering` | Joint expression + spatial |
